# Baseline Benchmark

Runs the initial LLM call on all benchmark queries, evaluates the results, and saves everything to disk (including `prompt.txt` per query so a refined run can load it later).

In [3]:
import importlib
import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from llm import call_llm, build_prompt
from analysis import analyze
import output as output_module

## Configuration

In [ ]:
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors
QUERY_LIMIT = 50        # set to None for all queries
MAX_WORKERS = 10
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Workers: {MAX_WORKERS} | Model: {MODEL}")

Loaded 11 benchmark sets (total available: 11)
Total queries: 3 (of 110 available)
Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro


## Prompt Template

In [ ]:
PROMPT_TEMPLATE = '''You're doing a Service Composition.
You are given a set of REST API specifications and a task description.
Your job is to write Python code using the appropriate client library that fulfills the task by calling the necessary endpoints in the correct order. Import requests and create a function called compose.

Rules:
- Use the requests library.
- Only use endpoints defined in the provided specifications.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- Import the requests library and create a function called compose, where all the requests shall be called. Do NOT call that function.

## Task
{query}

## Source
{services_block}

'''


## Initial LLM Call

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

tasks = []
for benchmark in benchmark_sets:
    if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
        break
    for query_index, query in enumerate(benchmark['queries'], start=1):
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        tasks.append((benchmark, query_index, query))

def _call_initial(args):
    benchmark, query_index, query = args
    prompt = build_prompt(benchmark['services'], query['query'], PROMPT_TEMPLATE)
    generated = call_llm(prompt, MODEL, '')
    generated += '\n\ncompose()'
    return {
        'query_index': query_index,
        'sector_name': benchmark['name'],
        'query': query,
        'prompt': prompt,
        'generated': generated,
        'service_files': benchmark.get('service_files', []),
        'model': MODEL
    }

sector_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_initial, t): t for t in tasks}
    for future in as_completed(futures):
        result = future.result()
        sector_results.append(result)
        print(f"[{len(sector_results)}/{total_queries}] [{result['sector_name']}] Query {result['query_index']} done")

sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))

[1/3] [01-energy] Query 2 done
[2/3] [01-energy] Query 1 done
[3/3] [01-energy] Query 3 done


## Evaluate

In [7]:
for result in sector_results:
    initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
    result['initial_metrics'] = initial_metrics
    print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")

Initial evaluation - Query 1
  Precision: 0.60
  Recall:    0.60
  F1:        0.60
  Extracted: ['GET /alert-settings', 'GET /equipment-monitoring', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   ['GET /alerts', 'GET /equipment-status']
  Extra:     ['GET /alert-settings', 'GET /equipment-monitoring']
Initial evaluation - Query 2
  Precision: 0.00
  Recall:    0.00
  F1:        0.00
  Extracted: []
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Extra:     []
Initial evaluation - Query 3
  Syntax error: invalid syntax at line 1, column 1
  Precis

## Save Outputs

In [8]:
from datetime import datetime
importlib.reload(output_module)
run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + "_baseline"
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': True,
    'model': MODEL,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-16_14-40-34_baseline


## Summary

In [9]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.20
  Average Recall:    0.20
  Average F1:        0.20
  Avg. Missing Endpoints: 5.00
  Avg. Extra Endpoints:   0.67
  Correct Compositions: 0/3 (0.0%)
